# Grounding DINO zero-shot object detection on match frames

This notebook loads a subset of frames from `../data/arsenal_mancity_frames_1fps`, runs zero-shot object detection with the Hugging Face `IDEA-Research/grounding-dino-tiny` model, and visualizes detections for the following concepts:

- player
- ref
- ball in play
- coach
- ball out of play

Notes:
- This is zero-shot: phrasing matters. You can tweak labels if results are poor (e.g., "a soccer player", "a referee", "a football").
- Detections are filtered by configurable score thresholds.
- Images are sampled evenly across the folder to cover the match timeline.

In [ ]:
# If running for the first time, ensure required packages are available
# Note: pip installs inside notebooks affect the current kernel environment

import transformers  # noqa: F401
import torch  # noqa: F401
import PIL  # noqa: F401
import requests  # noqa: F401

import os
import math
import glob
import requests
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    infer_device,
)

# Paths
DATA_DIR = Path("..") / "data" / "arsenal_mancity_frames_1fps"
OUTPUT_DIR = Path("..") / "data" / "export" / "grounding_dino"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading images from: {DATA_DIR}")
print(f"Saving annotated outputs to: {OUTPUT_DIR}")

In [ ]:
# Configure model and labels
model_id = "IDEA-Research/grounding-dino-base"
device = infer_device()
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

# Labels to detect (phrasing matters for zero-shot). Adjust/tune as needed.
# The outer list corresponds to a batch of prompts. We use a single prompt list here.
text_labels = [[
    "football",
    "soccer ball",
    "ball"
]]

# Thresholds
BOX_THRESHOLD = 0.40
TEXT_THRESHOLD = 0.30

In [ ]:
# Collect a sample of images across the directory timeline
image_paths = sorted(glob.glob(str(DATA_DIR / "*.jpg")))
if not image_paths:
    image_paths = sorted(glob.glob(str(DATA_DIR / "*.png")))

print(f"Found {len(image_paths)} candidate frames")

# Pick up to N samples evenly spaced
N_SAMPLES = 12
if len(image_paths) <= N_SAMPLES:
    sampled_paths = image_paths
else:
    step = len(image_paths) / N_SAMPLES
    sampled_paths = [image_paths[math.floor(i * step)] for i in range(N_SAMPLES)]

print(f"Sampling {len(sampled_paths)} frames for detection")
for p in sampled_paths[:3]:
    print(" -", p)

sampled_paths.append("/Users/georgebarnett/code/football-scan/tests/data/arsenal_mancity_test_detection.jpg")

In [ ]:
# Inference helpers
@torch.no_grad()
def run_grounded_detection(image: Image.Image):
    # The processor expects the text prompt to be a list of lists
    inputs = processor(images=image, text=text_labels, return_tensors="pt").to(model.device)
    outputs = model(**inputs)
    target_sizes = [image.size[::-1]]  # (height, width)
    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        target_sizes=target_sizes,
    )
    return results[0]

# Visualization helpers
PALETTE = {
    "football player person": (255, 0, 0),
    "football referee": (255, 165, 0),
    "ball in play": (0, 255, 0),
    "coach": (0, 128, 255),
    "ball out of play": (128, 0, 255),
    "person": (255, 0, 0),
}

try:
    FONT = ImageFont.truetype("DejaVuSans.ttf", 16)
except Exception:
    FONT = ImageFont.load_default()


def draw_detections(image: Image.Image, result: dict, min_score: float = 0.40) -> Image.Image:
    img = image.copy().convert("RGB")
    draw = ImageDraw.Draw(img)

    boxes = result.get("boxes", [])
    scores = result.get("scores", [])
    labels = result.get("labels", [])

    for box, score, label in zip(boxes, scores, labels):
        score = float(score)
        label = str(label)
        if score < min_score:
            continue
        x1, y1, x2, y2 = [float(x) for x in box.tolist()]
        color = PALETTE.get(label, (255, 255, 0))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = f"{label} {score:.2f}"
        tw, th = draw.textbbox((0, 0), text, font=FONT)[2:]
        draw.rectangle([x1, y1 - th - 4, x1 + tw + 6, y1], fill=color)
        draw.text((x1 + 3, y1 - th - 3), text, fill=(0, 0, 0), font=FONT)
    return img

In [ ]:
# Run detection and visualize in a grid, also save annotated images
num_cols = 3
num_rows = math.ceil(len(sampled_paths) / num_cols)
fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4.5 * num_rows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax in axes:
    ax.axis("off")

for idx, img_path in enumerate(sampled_paths):
    try:
        image = Image.open(img_path).convert("RGB")
        result = run_grounded_detection(image)
        annotated = draw_detections(image, result, min_score=BOX_THRESHOLD)

        # Show
        axes[idx].imshow(annotated)
        axes[idx].set_title(Path(img_path).name)

        # Save
        out_path = OUTPUT_DIR / f"annotated_{Path(img_path).stem}.jpg"
        annotated.save(out_path)
    except Exception as e:
        print(f"Failed on {img_path}: {e}")

plt.tight_layout()
plt.show()

print(f"Saved {len(sampled_paths)} annotated images to {OUTPUT_DIR}")

In [ ]:
Now run completely 